# Session 12: errors you expect, and `try` / `except`

Twice already the course has said "that needs `try` and `except`, and it is
session 12 material": the word typed where the menu wanted a number in session
3, and the empty list that crashes `average` in session 2. This is that
session.

The idea in one sentence: **some errors are not bugs, they are situations you
can predict**, and for those you can tell Python what to do instead of
crashing.

In [ ]:
import csv
import os
from pathlib import Path

# Walk up until we find the folder with the data/ directory in it.
here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

## Your four functions, back again

These are exactly the session 3 functions. Nothing has changed.

In [ ]:
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running


def average(numbers):
    """Return the mean of a list of numbers."""
    return total(numbers) / len(numbers)


def highest(numbers):
    """Return the largest value in a list of numbers."""
    best = numbers[0]
    for number in numbers:
        if number > best:
            best = number
    return best


def lowest(numbers):
    """Return the smallest value in a list of numbers."""
    worst = numbers[0]
    for number in numbers:
        if number < worst:
            worst = number
    return worst


print(average([12, 7, 41, 3, 28, 19, 7, 55, 2, 33]))

## The empty list, for real this time

In session 2 the empty list was hypothetical: you typed your own list, so it
was never empty. Here is how it happens in practice. Load the listening log
and ask for the average minutes per play on a device.

In [ ]:
def load_rows(path):
    """Return the CSV at path as a list of dictionaries, one per row."""
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def minutes_on(rows, device):
    """Return the minutes_played of every play on one device, as numbers."""
    return [float(row["minutes_played"]) for row in rows if row["device"] == device]


rows = load_rows("data/clean/plays.csv")
phone = minutes_on(rows, "phone")
print(f"{len(phone)} plays on the phone, {average(phone):.2f} minutes on average")

Now a device that is not in the data. A typo, or a device you simply never
used. The filter finds nothing, so the list is empty.

In [ ]:
radio = minutes_on(rows, "radio")
print(radio)

Uncomment and run it. Read the last line of the message first, as always.

In [ ]:
# average(radio)
# ZeroDivisionError: division by zero

Nothing is wrong with `average`. It was asked for the average of nothing,
and there is no such number. The crash is **predictable**: you can say in
advance exactly when it happens and which error it will be.

## `try` and `except`

Here is the shape. Python runs the `try` block. If a `ZeroDivisionError`
happens anywhere inside it, Python stops that block and jumps to the
`except` block instead. If nothing goes wrong, the `except` block is skipped.

In [ ]:
try:
    result = average(radio)
except ZeroDivisionError:
    result = None

print(result)

In [ ]:
try:
    result = average(phone)
except ZeroDivisionError:
    result = None

print(round(result, 2))

Same code, two lists. With the phone list nothing went wrong, so the
`except` line never ran.

`as error` gives you the error itself, if you want its message:

In [ ]:
try:
    average(radio)
except ZeroDivisionError as error:
    print("could not average:", error)

## Name the error. Never write a bare `except:`

Python lets you write `except:` with no error name, which catches
**everything**. It looks safer. It is the opposite. Watch what it does to a
typo:

In [ ]:
try:
    result = averag(radio)          # typo: averag, not average
except:
    result = None

print(result)

`None`, and no complaint. The real problem was a `NameError` (a misspelt
function), and the bare `except` swallowed it and pretended it was an empty
list. In a real program you would now spend an hour looking for a data
problem that does not exist.

With the error named, the typo still crashes, loudly, which is what you
want. Uncomment to see it:

In [ ]:
# try:
#     result = averag(radio)
# except ZeroDivisionError:
#     result = None
# NameError: name 'averag' is not defined

**The rule:** catch the one error you expected, by name, and let everything
else crash. An unexpected crash is information. A swallowed one is a lie
you will find later, at a worse moment.

The same goes for `except Exception:`, which catches nearly everything too.
You will see it in other people's code. Treat it as a question to ask in a
code review, not as a pattern to copy.

## Sometimes an `if` is better

You solved the empty list in session 3's stretch already, with no `try` at
all:

In [ ]:
def safe_average(numbers):
    """Return the mean of a list, or None if the list is empty."""
    if len(numbers) == 0:
        return None
    return average(numbers)


print(safe_average(radio), round(safe_average(phone), 2))

For an empty list that is the clearer version: you can check before you
leap, and the check reads like the sentence it stands for.

`try` earns its place when **checking in advance is awkward or impossible**.
Two cases come up constantly: text that may or may not be a number, and a
file that may or may not be there.

## Text that may not be a number: `ValueError`

Your session 3 menu needed to turn typed text into a number. Without `try`,
the worked solution had to do this, which it called "not elegant":

```python
if answer.lstrip("-").replace(".", "", 1).isdigit():
    return float(answer)
```

The honest way is to just try it:

In [ ]:
def to_number(text):
    """Return text as a float, or None if it is not a number."""
    try:
        return float(text)
    except ValueError:
        return None


assert to_number("21.5") == 21.5
assert to_number(" -3 ") == -3.0           # float() ignores surrounding spaces
assert to_number("1e3") == 1000.0          # the isdigit() version rejected this
assert to_number("abc") is None
assert to_number("") is None
print("checks passed")

Shorter, and more correct: `"1e3"` is a perfectly good number that the
`isdigit()` version turned away. Asking `float()` itself is the only check
that agrees with `float()` every time.

The menu's `ask_number` becomes:

```python
def ask_number(prompt):
    """Ask until the person types something that is actually a number."""
    while True:
        number = to_number(input(prompt))
        if number is not None:
            return number
        print("That does not look like a number. Have another go.")
```

### The same thing on real data

Session 9's messy file has a minutes column full of text like `"4.2 min"`
and empty strings. `to_number` sorts the good values from the bad in one
pass.

In [ ]:
with open("data/messy/plays_messy.csv", encoding="utf-8") as f:
    messy = list(csv.DictReader(f))

raw = [row["minutes played "] for row in messy]       # yes, that column name
numbers = [to_number(text) for text in raw]
good = [n for n in numbers if n is not None]
bad = [text for text, n in zip(raw, numbers) if n is None]

print(f"{len(raw)} values: {len(good)} numbers, {len(bad)} not numbers")
print("examples of the bad ones:", bad[:4])

assert (len(raw), len(good), len(bad)) == (2223, 2033, 190)

**Returning `None` is a decision, not a fix.** Those 190 values are 96
readings like `"4.2 min"`, which are perfectly recoverable, and 94 empty
strings, which are not. If you quietly drop all 190, every total you
report afterwards is smaller than the truth. That is session 9's lesson:
each cleaning decision changes the answer. `try` makes the decision easy to
write down. It does not make it for you.

In [ ]:
recoverable = [text for text in bad if text.endswith(" min")]
empty = [text for text in bad if text == ""]
print(len(recoverable), "like '4.2 min',", len(empty), "empty")
assert (len(recoverable), len(empty)) == (96, 94)

## A file that is not there: `FileNotFoundError`

The first error everybody meets in session 4. Here it is on purpose: one
letter missing from the file name.

In [ ]:
# load_rows("data/clean/play.csv")
# FileNotFoundError: [Errno 2] No such file or directory: 'data/clean/play.csv'

Could you check first, with `Path(...).exists()`? You could, and it is fine
for a quick script. But a file can disappear between the check and the
open, and a path can exist and still not be readable. `try` asks the only
question that matters: did opening it work?

**The interesting question is where to catch it.** Not here:

```python
def load_rows(path):
    try:
        with open(path, encoding="utf-8") as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        return []                # NO
```

That turns "the file is missing" into "the file has no plays in it". Every
number downstream is now a confident zero, or an average that crashes three
functions later with an error that has nothing to do with files. It is the
same lie as returning `0` for the average of nothing.

`load_rows` cannot do anything useful about a missing file, so it should
let the error travel up. The **outer layer** of the program, the part that
knows a person is waiting, catches it and says something helpful.

In [ ]:
def report(path):
    """The outer layer: load, summarise, and say what happened in words."""
    try:
        rows = load_rows(path)
    except FileNotFoundError:
        return (f"No file at {path}. Python is looking in {Path.cwd()}. "
                f"Check the spelling, and run from the repository root.")
    return f"{len(rows)} plays in {path}"


print(report("data/clean/plays.csv"))
print(report("data/clean/play.csv"))

That is the session 3 rule again, extended. **Functions that calculate
return; only the outer layer prints.** And now: functions that calculate let
errors travel; the outer layer decides what to do about them. The outer
layer is the only place that knows whether "file missing" means "tell the
person" or "stop the whole pipeline".

In a script, the outer layer is `main()`, and stopping with a message looks
like this:

```python
def main():
    try:
        rows = load_rows(PLAYS)
    except FileNotFoundError:
        raise SystemExit(f"No file at {PLAYS}. Check the name and the folder.")
    ...
```

`raise SystemExit(message)` prints the message and ends the program, without
a traceback, because there is nothing more to learn from one.

## The four errors worth knowing by name

| Error | You will see it when | Usually |
|---|---|---|
| `ZeroDivisionError` | dividing by a count that turned out to be zero | check with `if`, or catch it |
| `IndexError` | `numbers[0]` on an empty list | check with `if` first |
| `ValueError` | `float("4.2 min")`, `int("abc")` | catch it: trying is the only honest check |
| `FileNotFoundError` | a wrong path, or running from the wrong folder | catch it in the outer layer, with a helpful message |

And two you should almost never catch, because they mean your code is
wrong rather than your data: `NameError` (a typo) and `KeyError` on a
column name you mistyped. Let those crash, read them, fix the code.

## Next

`02-pytest.ipynb`: your session 3 `assert` lines, turned into real tests,
including a test that checks an error *does* happen.